# PD02 · Consultar y resumir EVA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es traducir una pregunta a filtros verificables y resúmenes. La pregunta de trabajo es: ¿cómo se distribuyen los registros agrícolas de Boyacá entre cultivos durante 2023 y 2024?

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD02_consultas_eva.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD02_consultas_eva.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 5, 8, 10 y las recetas de porcentajes y equivalencia de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD02"
archivos = ['eva.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD02


## 1. Preparación de columnas · 10 minutos

Inspeccionar la tabla y crear `codigo_departamento` a partir de `Código Dane departamento`, `anio` desde `Año`, `area_ha` desde `Área cosechada` y `produccion_t` desde `Producción`. El código se conserva como texto; año, área y producción se convierten a tipos numéricos. Registrar las ausencias resultantes.

📘 **Apoyo:** `.str.strip`, `pd.to_numeric`, `astype("Int64")` e `isna`.

✅ **Comprobación:** Las columnas originales se conservan y no se eliminan filas. El corte de entrada contiene 166.732 registros.

In [2]:
# Desarrollo de la actividad 1
# Preparar las cuatro columnas de trabajo en eva.
print(eva.shape)
print(list(eva.columns))
display(eva.head())

filas_originales = eva.shape[0]
columnas_originales = list(eva.columns)
print("=======================================================")

eva["codigo_departamento"] = eva["Código Dane departamento"].str.strip()
print(eva["codigo_departamento"].value_counts().head(10))
print(eva["codigo_departamento"].str.len().value_counts())
print("=======================================================")

eva["anio"] = pd.to_numeric(eva["Año"].str.strip(), errors="coerce").astype("Int64")
print(eva["anio"].min(), eva["anio"].max())
print("=======================================================")

eva["area_ha"] = pd.to_numeric(eva["Área cosechada"].str.strip(), errors="coerce")
eva["produccion_t"] = pd.to_numeric(eva["Producción"].str.strip(), errors="coerce")
print(eva[["area_ha", "produccion_t"]].describe())
print("=======================================================")

nuevas = ["codigo_departamento", "anio", "area_ha", "produccion_t"]
ausencias = pd.DataFrame({
    "columna": nuevas,
    "vacios_o_nulos": [eva[c].isna().sum() for c in nuevas],
})
display(ausencias)

print("Códigos vacíos:", eva["codigo_departamento"].eq("").sum())
print("=======================================================")
# Mostrar los tipos y los valores que no pudieron convertirse.

assert len(eva) == filas_originales == 166732
assert all(c in eva.columns for c in columnas_originales)

(166732, 18)
['Código Dane departamento', 'Departamento', 'Código Dane municipio', 'Municipio', 'Grupo cultivo', 'Subgrupo', 'Cultivo', 'Desagregación cultivo', 'Año', 'Periodo', 'Área sembrada', 'Área cosechada', 'Producción', 'Rendimiento', 'Ciclo del cultivo', 'Estado físico del cultivo', 'Código del cultivo', 'Nombre científico del cultivo']


,Código Dane departamento,Departamento,Código Dane municipio,Municipio,Grupo cultivo,Subgrupo,Cultivo,Desagregación cultivo,Año,Periodo,Área sembrada,Área cosechada,Producción,Rendimiento,Ciclo del cultivo,Estado físico del cultivo,Código del cultivo,Nombre científico del cultivo
0,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2021,2021A,8.00,8.00,128.00,16.00,Transitorio,En fresco,1050600,Apium graveolens
1,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2021,2021B,8.00,8.00,152.00,19.00,Transitorio,En fresco,1050600,Apium graveolens
2,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2022,2022A,8.00,8.00,127.44,15.93,Transitorio,En fresco,1050600,Apium graveolens
3,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2022,2022B,8.00,8.00,152.00,19.00,Transitorio,En fresco,1050600,Apium graveolens
4,05,Antioquia,05001,Medellín,Hortalizas,Hortalizas de tallo,Apio,Apio,2023,2023A,8.00,7.00,133.00,19.00,Transitorio,En fresco,1050600,Apium graveolens


codigo_departamento
15    17679
25    15584
05    15216
68    13071
41    11769
52    11568
76    10589
19     7518
54     7125
73     6688
Name: count, dtype: int64
codigo_departamento
2    166732
Name: count, dtype: int64
2019 2025
             area_ha  produccion_t
count  166732.000000  1.667320e+05
mean      208.991566  3.189636e+03
std      1120.053651  4.924617e+04
min         0.000000  0.000000e+00
25%         5.000000  2.000000e+01
50%        17.000000  1.000000e+02
75%        80.000000  5.490000e+02
max     77630.000000  4.776340e+06


,columna,vacios_o_nulos
0,codigo_departamento,0
1,anio,0
2,area_ha,0
3,produccion_t,0


Códigos vacíos: 0
166.732 filas y columnas originales conservadas


**Interpretación del resultado:**

Se crearon `codigo_departamento`, `anio`, `area_ha` y `produccion_t` a partir de las columnas originales, que se conservaron junto con las 166.732 filas de entrada.

`codigo_departamento` se mantuvo como texto: todos los códigos tienen dos caracteres y conservan el cero inicial, algo que se habría perdido con una conversión numérica. `anio` se convirtió a `Int64` y abarca de 2019 a 2025. `area_ha` y `produccion_t` se convirtieron a número con `pd.to_numeric(errors="coerce")`.

La tabla de ausencias muestra **0** valores ausentes en las cuatro columnas nuevas, y 0 códigos vacíos, por lo que todas las entradas fueron interpretables. Hay valores en **0** en área y producción, que provienen del archivo original. No se modificaron porque con los datos disponibles no es posible saber si son ceros reales o marcadores.

## 2. Selección con máscara y loc · 15 minutos

Seleccionar el departamento `15` (Boyacá), los años 2023 y 2024, área cosechada positiva y producción no negativa. Guardar todas las columnas de las filas seleccionadas en `consulta_loc`, utilizando `.copy()`. Presentar el total original, el seleccionado y la diferencia.

📘 **Apoyo:** `.eq`, `.isin`, `.gt`, `.ge`, `&` y `.loc`.

✅ **Comprobación:** Todas las filas seleccionadas cumplen simultáneamente las cuatro condiciones. La diferencia respecto a la entrada incluye registros fuera del alcance territorial o temporal; no implica que todos sean errores.

In [3]:
# Desarrollo de la actividad 2
# Construir la máscara y guardar consulta_loc.
m_depto = eva["codigo_departamento"].eq("15")
m_anio = eva["anio"].isin([2023, 2024])
m_area = eva["area_ha"].gt(0)
m_prod = eva["produccion_t"].ge(0)

print("Boyacá (15):", m_depto.sum())
print("Años 2023-2024:", m_anio.sum())
print("Área > 0:", m_area.sum())
print("Producción >= 0:", m_prod.sum())
print("==========================================")

mascara = m_depto & m_anio & m_area & m_prod
print("Filas que cumplen las cuatro:", mascara.sum())
print("==========================================")

consulta_loc = eva.loc[mascara].copy()
print(consulta_loc.shape)
display(consulta_loc.head())

resumen_consulta = pd.DataFrame({
    "concepto": ["Total original", "Seleccionado", "Diferencia"],
    "filas": [filas_originales, len(consulta_loc), filas_originales - len(consulta_loc)],
})
display(resumen_consulta)

# Mostrar los conteos de entrada, selección y diferencia.
assert consulta_loc["codigo_departamento"].eq("15").all()
assert consulta_loc["anio"].isin([2023, 2024]).all()
assert consulta_loc["area_ha"].gt(0).all()
assert consulta_loc["produccion_t"].ge(0).all()
assert len(consulta_loc) == mascara.sum()
assert len(eva) == filas_originales

Boyacá (15): 17679
Años 2023-2024: 50462
Área > 0: 159616
Producción >= 0: 166732
Filas que cumplen las cuatro: 4924
(4924, 22)


,Código Dane departamento,Departamento,Código Dane municipio,Municipio,Grupo cultivo,Subgrupo,Cultivo,Desagregación cultivo,Año,Periodo,...,Producción,Rendimiento,Ciclo del cultivo,Estado físico del cultivo,Código del cultivo,Nombre científico del cultivo,codigo_departamento,anio,area_ha,produccion_t
22910,15,Boyacá,15001,Tunja,Raíces y tubérculos,Raíces y tubérculos,Arracacha,Arracacha,2023,2023A,...,3750.00,25.00,Transitorio,En fresco,1080200,Arracacia xanthorrhiza,15,2023,150.0,3750.00
22912,15,Boyacá,15001,Tunja,Raíces y tubérculos,Raíces y tubérculos,Arracacha,Arracacha,2024,2024A,...,1250.00,25.00,Transitorio,En fresco,1080200,Arracacia xanthorrhiza,15,2024,50.0,1250.00
22913,15,Boyacá,15001,Tunja,Raíces y tubérculos,Raíces y tubérculos,Arracacha,Arracacha,2024,2024B,...,100.00,25.00,Transitorio,En fresco,1080200,Arracacia xanthorrhiza,15,2024,4.0,100.00
22924,15,Boyacá,15001,Tunja,Leguminosas,Leguminosas,Arveja,Arveja,2023,2023A,...,29.44,1.64,Transitorio,Grano,1060100,Pisum sativum,15,2023,18.0,29.44
22925,15,Boyacá,15001,Tunja,Leguminosas,Leguminosas,Arveja,Arveja,2023,2023B,...,747.66,1.87,Transitorio,Grano,1060100,Pisum sativum,15,2023,400.0,747.66


,concepto,filas
0,Total original,166732
1,Seleccionado,4924
2,Diferencia,161808


Todas las filas seleccionadas cumplen las cuatro condiciones


**Interpretación del resultado:**

Se seleccionaron con máscaras booleanas y `.loc` las filas de Boyacá (`codigo_departamento == "15"`), de los años 2023 y 2024, con área cosechada positiva y producción no negativa. El resultado se guardó con todas las columnas en `consulta_loc` (4.924 filas y 22 columnas), usando `.copy()` para trabajar sobre una tabla independiente de `eva`.

De las 166.732 filas de entrada se seleccionaron **4.924**, y la diferencia es de **161.808**. Los `assert` confirman que todas las filas seleccionadas cumplen las cuatro condiciones a la vez.

## 3. Consulta equivalente con query · 10 minutos

Repetir exactamente la selección anterior mediante `.query`, con `engine="python"`, y guardar el resultado en `consulta_query`. Usar una variable para el código departamental y referenciarla con `@`. Comparar ambos DataFrames sin eliminar columnas.

📘 **Apoyo:** `.query`, `@variable` y `pd.testing.assert_frame_equal`.

✅ **Comprobación:** La comparación debe confirmar las mismas filas, columnas y valores. Si falla, se revisan las condiciones y los tipos antes de continuar.

In [4]:
# Desarrollo de la actividad 3
# Construir consulta_query.
cod_depto = "15"
print(cod_depto)

consulta_query = eva.query(
    "codigo_departamento == @cod_depto and anio in [2023, 2024] and area_ha > 0 and produccion_t >= 0",
    engine="python",
).copy()
print(consulta_query.shape)

print("loc:  ", consulta_loc.shape)
print("query:", consulta_query.shape)

# Comprobar que consulta_loc y consulta_query son equivalentes.
pd.testing.assert_frame_equal(consulta_loc, consulta_query)
print("consulta_loc y consulta_query son equivalentes")

assert list(consulta_query.columns) == list(eva.columns)
assert len(consulta_query.columns) == 22
print("22 columnas conservadas")

15
(4924, 22)
loc:   (4924, 22)
query: (4924, 22)
consulta_loc y consulta_query son equivalentes
22 columnas conservadas


**Interpretación del resultado:**

Se repitió la selección de la actividad 2 con `.query` (`engine="python"`), referenciando el código departamental mediante la variable `cod_depto` con `@`. El resultado se guardó en `consulta_query`.

La comparación con `pd.testing.assert_frame_equal` confirmó que ambas tablas tienen las mismas filas, columnas, tipos, índice y valores. Esto muestra que las máscaras booleanas con `.loc` y la expresión de `.query` son dos formas equivalentes de expresar la misma selección.

## 4. Conteos y porcentajes · 20 minutos

Sobre `consulta_loc`, construir `resumen` con una fila por año y cultivo y una columna `registros`. Añadir el total de registros de cada año y `porcentaje_anual`. Ordenar los resultados por año y cantidad de registros descendente. Conservar los grupos con claves ausentes mediante `dropna=False`.

📘 **Apoyo:** `groupby`, `size`, `reset_index`, `transform("sum")` y `sort_values`.

✅ **Comprobación:** La suma de registros coincide con el tamaño de consulta_loc. Los porcentajes suman aproximadamente 100 % por año; se admite una pequeña tolerancia numérica y se redondea solo para mostrar.

In [7]:
# Desarrollo de la actividad 4
# Crear resumen con registros, total_anio y porcentaje_anual.
resumen = (
    consulta_loc
    .groupby(["anio", "Cultivo"], dropna=False)
    .size()
    .reset_index(name="registros")
)
print(resumen.shape)
display(resumen.head())

resumen["total_anio"] = resumen.groupby("anio", dropna=False)["registros"].transform("sum")
display(resumen.head())

resumen["porcentaje_anual"] = resumen["registros"] / resumen["total_anio"] * 100

resumen = resumen.sort_values(
    ["anio", "registros", "Cultivo"],
    ascending=[True, False, True],
).reset_index(drop=True)
display(resumen.head(10).round({"porcentaje_anual": 2}))

# Verificar los totales y los porcentajes por año.
assert resumen["registros"].sum() == len(consulta_loc)

suma_pct = resumen.groupby("anio")["porcentaje_anual"].sum()
print(suma_pct)
assert (suma_pct - 100).abs().lt(1e-9).all()

print("Registros concilian con consulta_loc:", resumen["registros"].sum())

(192, 3)


,anio,Cultivo,registros
0,2023,Acelga,5
1,2023,Achira,4
2,2023,Agraz - mortiño,3
3,2023,Aguacate,48
4,2023,Ahuyama,23


,anio,Cultivo,registros,total_anio
0,2023,Acelga,5,2436
1,2023,Achira,4,2436
2,2023,Agraz - mortiño,3,2436
3,2023,Aguacate,48,2436
4,2023,Ahuyama,23,2436


,anio,Cultivo,registros,total_anio,porcentaje_anual
0,2023,Papa,239,2436,9.81
1,2023,Maíz,194,2436,7.96
2,2023,Arveja,176,2436,7.22
3,2023,Frijol,162,2436,6.65
4,2023,Tomate,113,2436,4.64
5,2023,Cebolla de bulbo,93,2436,3.82
6,2023,Yuca,82,2436,3.37
7,2023,Caña,63,2436,2.59
8,2023,Trigo,58,2436,2.38
9,2023,Mora,51,2436,2.09


anio
2023    100.0
2024    100.0
Name: porcentaje_anual, dtype: float64
Registros concilian con consulta_loc: 4924


**Interpretación del resultado:**

Sobre `consulta_loc` se construyó `resumen` con una fila por año y cultivo, con las columnas `registros`, `total_anio` y `porcentaje_anual`. Se usó `dropna=False` para conservar grupos con claves ausentes, y el orden es por año ascendente y registros descendente.

Los **4.924** registros de `resumen` concilian con el tamaño de `consulta_loc`, y los porcentajes suman 100 % en cada año (2023 y 2024), con una tolerancia numérica mínima. El redondeo se aplicó solo para mostrar.

## 5. Tabla dinámica y conclusiones · 15 minutos

Construir una tabla dinámica a partir de `resumen`, con cultivos en filas, años en columnas y conteos como valores. Completar con cero las combinaciones sin registros. Exportar `resumen` a `OUT / "resumen_eva.csv"`, sin índice, y redactar dos conclusiones.

📘 **Apoyo:** `pivot_table`, `fill_value=0` y `to_csv`.

✅ **Comprobación:** Los totales de la tabla dinámica concilian con resumen. Las conclusiones se refieren a la cantidad de registros y no a la participación de cada cultivo en la superficie agrícola.

In [8]:
# Desarrollo de la actividad 5
# Construir y mostrar la tabla dinámica.
tabla_dinamica = resumen.pivot_table(
    index="Cultivo",
    columns="anio",
    values="registros",
    aggfunc="sum",
    fill_value=0,
)
print(tabla_dinamica.shape)
display(tabla_dinamica.head(10))

assert tabla_dinamica.to_numpy().sum() == resumen["registros"].sum()
print("Total tabla dinámica:", tabla_dinamica.to_numpy().sum())
print("Total resumen:       ", resumen["registros"].sum())
print("Totales por año:")
print(tabla_dinamica.sum())

sin_registros = (tabla_dinamica == 0).sum()
print("Cultivos con 0 registros por año:")
print(sin_registros)
display(tabla_dinamica[(tabla_dinamica == 0).any(axis=1)].head())

# Exportar resumen y explicar dos resultados en la celda siguiente.
ruta_resumen = OUT / "resumen_eva.csv"
resumen.to_csv(ruta_resumen, index=False)

assert ruta_resumen.is_file() and ruta_resumen.parent == OUT
print("Archivo generado:", ruta_resumen)
display(pd.read_csv(ruta_resumen).head())

tabla_dinamica["total"] = tabla_dinamica.sum(axis=1)
display(tabla_dinamica.sort_values("total", ascending=False).head(5))
tabla_dinamica = tabla_dinamica.drop(columns="total")

(97, 2)


anio,2023,2024
Cultivo,,
Acelga,5,8
Achira,4,4
Agraz - mortiño,3,4
Aguacate,48,52
Ahuyama,23,26
Ajo,7,7
Ají,1,2
Apio,0,1
Arazá,3,4


Total tabla dinámica: 4924
Total resumen:        4924
Totales por año:
anio
2023    2436
2024    2488
dtype: int64
Cultivos con 0 registros por año:
anio
2023    1
2024    1
dtype: int64


anio,2023,2024
Cultivo,,
Apio,0,1
Mangostino,1,0


Archivo generado: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD02/resumen_eva.csv


,anio,Cultivo,registros,total_anio,porcentaje_anual
0,2023,Papa,239,2436,9.811166
1,2023,Maíz,194,2436,7.963875
2,2023,Arveja,176,2436,7.224959
3,2023,Frijol,162,2436,6.650246
4,2023,Tomate,113,2436,4.638752


anio,2023,2024,total
Cultivo,,,
Papa,239,237,476
Maíz,194,196,390
Arveja,176,168,344
Frijol,162,169,331
Tomate,113,120,233


**Interpretación del resultado:**

Se construyó una tabla dinámica a partir de `resumen`, con cultivos en filas, años en columnas y la suma de `registros` como valores, completando con cero las combinaciones sin registros. Sus totales concilian con `resumen`: **4.924** registros en total, **2.436** en 2023 y **2.488** en 2024. `resumen` se exportó sin índice a `OUT / "resumen_eva.csv"`.

**Conclusiones:**

1. **Los registros se concentran en pocos cultivos y se mantienen estables entre años.** Papa (476 registros en los dos años), maíz (390), arveja (344), frijol (331) y tomate (233) encabezan la tabla, y sus conteos casi no cambian de 2023 a 2024. Esto indica que, en Boyacá, estos cultivos son los que más veces aparecen reportados en la EVA con área cosechada positiva. Se refiere a la cantidad de registros (filas municipio-cultivo-periodo), no a la superficie que ocupa cada cultivo.

## Entrega

El notebook completado se conserva como `soluciones/PD02_consultas_eva.ipynb`. La salida regenerable es `kit/salidas/soluciones/PD02/resumen_eva.csv`.

✅ Deben quedar visibles la equivalencia de consultas, la conciliación de los conteos, los porcentajes por año, la tabla dinámica y dos conclusiones.

⚠️ Una fila EVA es una observación de la fuente, no un predio ni una persona. Un porcentaje de registros no equivale a un porcentaje de área cultivada. El cero de una tabla de conteos indica que no se encontraron filas para esa combinación.